In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\abdul\AppData\Local\Temp\ipykernel_10828\1266949206.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
load_dotenv()  # loads BLABLADOR_API_KEY from .env

True

In [3]:
# Load all .md files from your local docs folder
loader = DirectoryLoader("./docs", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
docs = loader.load()
print(f"Loaded {len(docs)} documents.")

Loaded 123 documents.


In [4]:
base_url= "https://api.blablador.fz-juelich.de/v1"
api_key=os.getenv("BLABLADOR_API_KEY")

In [5]:
# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

# Embed and persist — this replaces VectorstoreIndexCreator entirely
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="alias-embeddings", api_key=api_key, base_url=base_url),
    persist_directory="./chroma_db"
)
vectorstore.persist()
print(f"Indexed {len(chunks)} chunks")

Indexed 797 chunks


C:\Users\abdul\AppData\Local\Temp\ipykernel_10828\3606625362.py:11: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [6]:
question = "What are the UFZ guidelines for long-term archiving?"
docs = vectorstore.similarity_search(question, k=4)
context = "\n\n".join(d.page_content for d in docs)

In [7]:
llm = ChatOpenAI(model="alias-fast", api_key=api_key, base_url=base_url)
answer = llm.invoke(f"Answer based on this context:\n\n{context}\n\nQuestion: {question}")
print(answer.content)

**UFZ (Leibniz‑Centre for Agricultural Landscape Research) – Guidelines for Long‑Term Archiving**

The UFZ has codified a set of “best‑practice” requirements that any research data set must meet before it can be deposited in a long‑term archive.  The core of the guidelines is centred on **metadata quality, data provenance and technical sustainability** – exactly the elements that guarantee that the data can be reused and that its scientific results remain reproducible.

Below is a concise, step‑by‑step summary of the UFZ recommendations (the full text can be found in the UFZ‑Data‑Management‑Policy and the associated “Research Data Management” handbook).

---

### 1.  Metadata – the backbone of archiving  

| Requirement | What UFZ expects | Why it matters |
|------------|------------------|----------------|
| **Standardised technical metadata** | Use an accepted schema (e.g., **Dublin Core**, **DataCite**, **ISA‑Tab**, or UFZ‑specific extensions). The metadata file must be machine‑read

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question = "What are the UFZ guidelines for long-term archiving?"

docs = retriever.invoke(question)
context = "\n\n".join(d.page_content for d in docs)

answer = llm.invoke(f"Answer based on this context:\n\n{context}\n\nQuestion: {question}")
print(answer.content)

**UFZ (Helmholtz‑Centre for Environmental Research) – Guidelines for Long‑Term Archiving**  
*(derived from the UFZ Data‑Management Handbook, the “FAIR‑Data” policy and the “Research Data Management (RDM) – Long‑Term Preservation” instruction)*  

---

## 1. Core Principles  

| Principle | What UFZ expects | Why it matters |
|-----------|------------------|----------------|
| **FAIR** (Findable, Accessible, Interoperable, Reusable) | All archived datasets must be **findable** via persistent identifiers (DOI/URN), **accessible** under clear usage licences, **interoperable** through standardised formats and vocabularies, and **re‑usable** with complete, machine‑readable metadata. | Guarantees that the data can be discovered and used by anyone (including future generations) and that the scientific record remains reproducible. |
| **Sustainability** | Data must be stored on UFZ‑approved, **institutional storage systems** (e.g. the UFZ Datenplattform/DSpace, the High‑Performance‑Computing 